In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "civic_signal_dbx_dev"

WATERMARK_TABLE = f"{CATALOG}.ops.silver_watermarks"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE}
(
    pipeline_name STRING,
    last_ingest_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Operational watermarks used to incrementally process new Bronze records into Civic Signal Silver tables.'
""")

DataFrame[]

In [0]:
def get_watermark(pipeline_name):
    rows = (
        spark.table(WATERMARK_TABLE)
        .filter(F.col("pipeline_name") == pipeline_name)
        .select("last_ingest_timestamp")
        .collect()
    )

    if not rows:
        return "1900-01-01 00:00:00"

    return rows[0]["last_ingest_timestamp"]


def set_watermark(pipeline_name, watermark):
    spark.sql(f"""
    MERGE INTO {WATERMARK_TABLE} AS t
    USING (
        SELECT
            '{pipeline_name}' AS pipeline_name,
            TIMESTAMP('{watermark}') AS last_ingest_timestamp
    ) AS s
    ON t.pipeline_name = s.pipeline_name

    WHEN MATCHED THEN
      UPDATE SET t.last_ingest_timestamp = s.last_ingest_timestamp

    WHEN NOT MATCHED THEN
      INSERT (pipeline_name, last_ingest_timestamp)
      VALUES (s.pipeline_name, s.last_ingest_timestamp)
    """)

In [0]:
opp_watermark = get_watermark("silver_opportunities")

opp_bronze = (
    spark.table(f"{CATALOG}.bronze.opportunities_raw")
    .filter(F.col("_ingest_timestamp") > F.lit(opp_watermark))
)

opp_contract = opp_bronze.select(
    "opportunity_id",
    "opportunity_title",
    "buyer_id",
    "category_code",
    "country",
    "source_system",
    "procurement_method",
    "estimated_value",
    "currency",
    "published_at",
    "closing_at",
    "status",
    "updated_at",
    "_rescued_data",
    "_ingest_timestamp",
    "_source_file",
    "_source_file_modification_time"
)


print("New Bronze opportunity rows:", opp_bronze.count())
print("Previous watermark:", opp_watermark)

New Bronze opportunity rows: 0
Previous watermark: 2026-09-20 20:00:30.629000


In [0]:
opp_clean = (
    opp_contract
    .withColumn("opportunity_id", F.trim("opportunity_id"))
    .withColumn("opportunity_title", F.trim("opportunity_title"))
    .withColumn("buyer_id", F.trim("buyer_id"))
    .withColumn("category_code", F.trim("category_code"))
    .withColumn("country", F.trim("country"))
    .withColumn("source_system", F.trim("source_system"))
    .withColumn("procurement_method", F.trim("procurement_method"))
    .withColumn("currency", F.trim("currency"))
    .withColumn("status", F.trim("status"))

    .withColumn("published_at", F.to_timestamp("published_at"))
    .withColumn("closing_at", F.to_timestamp("closing_at"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    .withColumn("estimated_value", F.col("estimated_value").cast("decimal(18,2)"))

    .withColumn(
        "dq_reason",
        F.when(
            F.col("opportunity_id").isNull() |
            (F.length("opportunity_id") == 0),
            F.lit("MISSING_OPPORTUNITY_ID")
        )
        .when(
            F.col("opportunity_title").isNull() |
            (F.length("opportunity_title") == 0),
            F.lit("MISSING_TITLE")
        )
        .when(
            F.col("buyer_id").isNull() |
            (F.length("buyer_id") == 0),
            F.lit("MISSING_BUYER_ID")
        )
        .when(
            F.col("published_at").isNull(),
            F.lit("INVALID_PUBLISHED_AT")
        )
        .when(
            F.col("estimated_value") < 0,
            F.lit("NEGATIVE_ESTIMATED_VALUE")
        )
    )

    .withColumn(
        "record_status",
        F.when(F.col("dq_reason").isNotNull(), F.lit("QUARANTINE"))
         .otherwise(F.lit("VALID"))
    )
)

In [0]:
opp_window = (
    Window
    .partitionBy("opportunity_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last(),
        F.col("_ingest_timestamp").desc(),
        F.col("_source_file_modification_time").desc()
    )
)

opp_latest = (
    opp_clean
    .withColumn("_rn", F.row_number().over(opp_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

In [0]:
opp_valid = (
    opp_latest
    .filter(F.col("record_status") == "VALID")
)

opp_quarantine = (
    opp_latest
    .filter(F.col("record_status") == "QUARANTINE")
)

In [0]:
opp_target = f"{CATALOG}.silver.opportunities"

if not spark.catalog.tableExists(opp_target):

    (
        opp_valid.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(opp_target)
    )

else:

    target = DeltaTable.forName(spark, opp_target)

    (
        target.alias("t")
        .merge(
            opp_valid.alias("s"),
            "t.opportunity_id = s.opportunity_id"
        )
        .whenMatchedUpdateAll(
            condition="s.updated_at > t.updated_at"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
opp_quarantine_target = f"{CATALOG}.silver.opportunities_quarantine"

if not spark.catalog.tableExists(opp_quarantine_target):
    (
        opp_quarantine.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(opp_quarantine_target)
    )
elif opp_quarantine.count() > 0:
    (
        opp_quarantine.write
        .format("delta")
        .mode("append")
        .saveAsTable(opp_quarantine_target)
    )

In [0]:
buyer_watermark = get_watermark("silver_buyers")

buyer_bronze = (
    spark.table(f"{CATALOG}.bronze.buyers_raw")
    .filter(F.col("_ingest_timestamp") > F.lit(buyer_watermark))
)

print("New Bronze buyer rows:", buyer_bronze.count())

buyer_clean = (
    buyer_bronze
    .withColumn("buyer_id", F.trim("buyer_id"))
    .withColumn("buyer_name", F.trim("buyer_name"))
    .withColumn("buyer_type", F.trim("buyer_type"))
    .withColumn("country", F.trim("country"))
    .withColumn("region", F.trim("region"))
    .withColumn("source_system", F.trim("source_system"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))

    .withColumn(
        "record_status",
        F.when(
            F.col("buyer_id").isNull() |
            (F.length("buyer_id") == 0) |
            F.col("buyer_name").isNull() |
            (F.length("buyer_name") == 0),
            F.lit("QUARANTINE")
        ).otherwise(F.lit("VALID"))
    )
)

buyer_window = (
    Window
    .partitionBy("buyer_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last(),
        F.col("_ingest_timestamp").desc()
    )
)

buyer_latest = (
    buyer_clean
    .withColumn("_rn", F.row_number().over(buyer_window))
    .filter("_rn = 1")
    .drop("_rn")
)

buyer_valid = buyer_latest.filter("record_status = 'VALID'")

buyer_target = f"{CATALOG}.silver.buyers"

if not spark.catalog.tableExists(buyer_target):

    (
        buyer_valid.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(buyer_target)
    )

else:

    target = DeltaTable.forName(spark, buyer_target)

    (
        target.alias("t")
        .merge(
            buyer_valid.alias("s"),
            "t.buyer_id = s.buyer_id"
        )
        .whenMatchedUpdateAll(
            condition="s.updated_at > t.updated_at"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

New Bronze buyer rows: 0


In [0]:
categories_silver = (
    spark.table(f"{CATALOG}.bronze.categories_raw")
    .select(
        F.trim("category_code").alias("category_code"),
        F.trim("category_name").alias("category_name"),
        F.trim("category_group").alias("category_group")
    )
    .dropDuplicates(["category_code"])
)

(
    categories_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.categories")
)

In [0]:
opp_max_ingest = (
    opp_bronze
    .agg(F.max("_ingest_timestamp").alias("max_ts"))
    .first()["max_ts"]
)

buyer_max_ingest = (
    buyer_bronze
    .agg(F.max("_ingest_timestamp").alias("max_ts"))
    .first()["max_ts"]
)

if opp_max_ingest:
    set_watermark("silver_opportunities", opp_max_ingest)

if buyer_max_ingest:
    set_watermark("silver_buyers", buyer_max_ingest)

In [0]:
spark.sql("""
COMMENT ON TABLE civic_signal_dbx_dev.silver.opportunities IS
'Current-state synthetic procurement opportunities standardized, validated and incrementally merged from Bronze using opportunity_id and updated_at.'
""")

spark.sql("""
COMMENT ON TABLE civic_signal_dbx_dev.silver.opportunities_quarantine IS
'Synthetic procurement opportunity records rejected from the Silver current-state dataset because they failed defined data-quality requirements.'
""")

spark.sql("""
COMMENT ON TABLE civic_signal_dbx_dev.silver.buyers IS
'Current-state synthetic public buyer entities standardized and incrementally merged from Bronze using buyer_id and updated_at.'
""")

spark.sql("""
COMMENT ON TABLE civic_signal_dbx_dev.silver.categories IS
'Standardized synthetic procurement category reference data used by the Civic Signal analytical model.'
""")

DataFrame[]

In [0]:
tables = [
    "opportunities",
    "opportunities_quarantine",
    "buyers",
    "categories"
]

for table in tables:
    count = spark.table(
        f"{CATALOG}.silver.{table}"
    ).count()

    print(f"{table}: {count}")

opportunities: 11
opportunities_quarantine: 0
buyers: 6
categories: 7


In [0]:
display(
    spark.table(f"{CATALOG}.silver.opportunities")
    .filter(
        F.col("opportunity_id").isin(
            "CIV-OPP-002",
            "CIV-OPP-003"
        )
    )
    .select(
        "opportunity_id",
        "estimated_value",
        "closing_at",
        "updated_at"
    )
)

opportunity_id,estimated_value,closing_at,updated_at
CIV-OPP-002,450000.00,2026-10-02T14:00:00.000Z,2026-09-10T10:30:00.000Z
CIV-OPP-003,100000.00,2026-09-27T12:00:00.000Z,2026-09-10T11:00:00.000Z


In [0]:
display(
    spark.table("civic_signal_dbx_dev.silver.buyers")
    .filter(F.col("buyer_id") == "CIV-BUY-003")
    .select(
        "buyer_id",
        "buyer_name",
        "updated_at"
    )
)

buyer_id,buyer_name,updated_at
CIV-BUY-003,Civic Kenya Digital Technology Authority,2026-09-10T07:10:00.000Z
